# ETL / Data Wrangling — Reference & Quiz Notes
> **Level:** Intermediate | **Goal:** Master ETL concepts, pipelines, testing, and Python implementation

## Table of Contents
1. [What is ETL?](#etl-intro)
2. [ETL vs ELT](#etl-vs-elt)
3. [Data Sources for ETL](#sources)
4. [ETL Testing](#testing)
5. [Incremental Loads](#incremental)
6. [Outlier Replacement Pipeline](#outliers)
7. [Kafka Streaming ETL](#kafka)
8. [SQL in ETL — JOIN & Aggregation](#sql-etl)
9. [Full Python ETL Pipeline](#full-pipeline)

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Setup complete ✅')

---
## 1 · What is ETL? <a id='etl-intro'></a>

**ETL = Extract → Transform → Load**

| Phase | What happens | Tools |
|---|---|---|
| **Extract** | Pull raw data from source systems | SQL, APIs, file readers, Kafka consumers |
| **Transform** | Clean, reshape, validate, enrich | pandas, dbt, Spark, KStreams |
| **Load** | Write processed data to target | DB inserts, Parquet writes, warehouse loaders |

### Common data sources (Extract)

| Source | Example | Common? |
|---|---|---|
| **Flat files** | CSV, JSON, XML, Parquet | ✅ Very common |
| **Relational databases** | PostgreSQL, MySQL, SQLite | ✅ Very common |
| **Web pages** | Web scraping, HTML | ✅ Common |
| **APIs** | REST, GraphQL | ✅ Very common |
| **Message queues** | Kafka, RabbitMQ | ✅ Streaming |
| **Data warehouses** | Snowflake, BigQuery | ❌ **Not a source — this is the TARGET** |

> 📝 **Quiz answer:** Data warehouses are **NOT** a common source for ETL — they are the **destination**.

---
## 2 · ETL vs ELT <a id='etl-vs-elt'></a>

| | ETL | ELT |
|---|---|---|
| **Order** | Extract → Transform → Load | Extract → Load → Transform |
| **Where transform happens** | Before loading (staging area) | Inside the data warehouse |
| **Best for** | Legacy systems, sensitive data | Cloud DWH (BigQuery, Snowflake, Redshift) |
| **Tools** | Informatica, Talend, pandas | dbt, Spark SQL, BigQuery SQL |

```
ETL:  [Source] → Extract → Transform (staging) → Load → [Data Warehouse]
ELT:  [Source] → Extract → Load (raw) → [Data Warehouse] → Transform (SQL)
```

---
## 3 · Data Sources for ETL <a id='sources'></a>

### Quiz question
> *"Which is NOT a common data source to feed an ETL?"*  
> ✅ **Data warehouses** — DWH is the **target/destination**, not the source

| Source type | Read with | Example |
|---|---|---|
| CSV / flat files | `pd.read_csv()` | Sales exports, logs |
| Relational DB | `pd.read_sql()` / `sqlite3` | Production DB tables |
| JSON / APIs | `pd.read_json()` / `requests` | REST API responses |
| Parquet | `pd.read_parquet()` | Data lake files |
| Excel | `pd.read_excel()` | Business reports |
| Web scraping | `BeautifulSoup` / `Scrapy` | HTML pages |

In [ ]:
# ── Reading from multiple source types ─────────────────────────
import io, json

# 1. From CSV string (simulating a flat file)
csv_data = """order_id,date,revenue
1001,2024-01-05,250.0
1002,2024-01-06,180.5
1003,2024-01-07,320.0"""
df_csv = pd.read_csv(io.StringIO(csv_data))
print('── From flat file (CSV) ──')
display(df_csv)

# 2. From JSON (simulating an API response)
json_data = '[{"order_id":1001,"product":"Laptop","qty":2},{"order_id":1002,"product":"Mouse","qty":5}]'
df_json = pd.read_json(io.StringIO(json_data))
print('── From JSON / API ──')
display(df_json)

# 3. From relational database (SQLite)
conn = sqlite3.connect(':memory:')
df_csv.to_sql('orders', conn, index=False, if_exists='replace')
df_db = pd.read_sql('SELECT * FROM orders WHERE revenue > 200', conn)
print('── From relational DB (SQLite) ──')
display(df_db)

---
## 4 · ETL Testing <a id='testing'></a>

### Quiz question
> *"Which are the main purposes of ETL testing?"*  
> ✅ **All of the above:**
> 1. Verify the **accuracy and completeness** of data in the target system
> 2. Validate the **functionality and performance** of ETL tools and technologies
> 3. Check the **quality and consistency** of data throughout the ETL process

### ETL testing checklist

| Test type | What it checks | Example |
|---|---|---|
| **Completeness** | All rows transferred | `len(source) == len(target)` |
| **Accuracy** | Values are correct | `source['revenue'].sum() == target['revenue'].sum()` |
| **Consistency** | No duplicates | `target['order_id'].is_unique` |
| **Validity** | No nulls in required fields | `target['order_id'].notna().all()` |
| **Referential integrity** | FK values exist in parent | All `customer_id` values exist in customers table |
| **Performance** | Pipeline runs within SLA | Time the pipeline run |

In [ ]:
# ── ETL testing framework ───────────────────────────────────────
def run_etl_tests(source_df, target_df, id_col='order_id', value_col='revenue'):
    """Run a suite of ETL data quality tests."""
    tests = {}

    # 1. Completeness
    tests['row_count_match'] = len(source_df) == len(target_df)

    # 2. Accuracy — numeric totals match
    if value_col in source_df and value_col in target_df:
        tests['revenue_sum_match'] = abs(
            source_df[value_col].sum() - target_df[value_col].sum()) < 0.01

    # 3. Consistency — no duplicates
    if id_col in target_df:
        tests['no_duplicates'] = target_df[id_col].is_unique

    # 4. Validity — no nulls in key columns
    if id_col in target_df:
        tests['no_null_ids'] = target_df[id_col].notna().all()

    # 5. No nulls in value column
    if value_col in target_df:
        tests['no_null_revenue'] = target_df[value_col].notna().all()

    # Report
    print('── ETL Test Results ──────────────────')
    all_pass = True
    for test, result in tests.items():
        status = '✅ PASS' if result else '❌ FAIL'
        print(f'  {test:30s}: {status}')
        if not result: all_pass = False
    print(f"\n  Overall: {'✅ ALL TESTS PASSED' if all_pass else '❌ SOME TESTS FAILED'}")
    return all_pass

# Demo
source = pd.DataFrame({'order_id': [1,2,3,4,5], 'revenue': [100,200,300,400,500]})
target_good = source.copy()
target_bad  = source.copy()
target_bad.loc[5] = [5, None]  # introduce a null
target_bad = pd.concat([target_bad, pd.DataFrame([{'order_id':1,'revenue':100}])], ignore_index=True)  # duplicate

print('=== Good target ===')
run_etl_tests(source, target_good)

print('\n=== Bad target (null + duplicate) ===')
run_etl_tests(source, target_bad)

---
## 5 · Incremental Loads <a id='incremental'></a>

### Quiz question
> *"Best way to implement incremental loads in an ETL pipeline?"*  
> ✅ **Use a timestamp column** (`updated_at`) to filter only new/modified records

### Incremental load strategies

| Strategy | How | Best for |
|---|---|---|
| **Timestamp-based** ✅ | `WHERE updated_at > last_run` | Tables with reliable `updated_at` |
| **High-water mark** | Track max ID or date processed | Append-only tables |
| **CDC (Change Data Capture)** | Read DB transaction logs | Full change tracking |
| **Partition-based** | Process only today's partition | Data lakes |

> A **message queue** (Kafka, RabbitMQ) is for **streaming**, not incremental batch loads.

In [ ]:
# ── Incremental load pattern ────────────────────────────────────
import datetime

# Simulate source table with timestamps
source_db = pd.DataFrame({
    'order_id':   range(1, 11),
    'revenue':    [100,200,150,300,250,180,220,400,130,290],
    'updated_at': pd.date_range('2024-01-01', periods=10, freq='D')
})

# Full load (first run)
last_successful_run = pd.Timestamp('2024-01-01')

def incremental_extract(source, last_run):
    """Extract only records updated after last_run."""
    new_records = source[source['updated_at'] > last_run].copy()
    return new_records

print(f'Source table: {len(source_db)} rows')
display(source_db)

print(f'\n── Incremental extract (last_run = {last_successful_run.date()}) ──')
incremental = incremental_extract(source_db, last_successful_run)
print(f'Records to process: {len(incremental)} (only new/updated)')
display(incremental)

# After processing, update the watermark
new_watermark = incremental['updated_at'].max()
print(f'\nNew watermark for next run: {new_watermark.date()}')

---
## 6 · Outlier Replacement Pipeline <a id='outliers'></a>

### Task
Replace revenue outliers using percentile-based winsorizing:
- Outliers = values **below P1** or **above P99**  
- Replace above P99 with **max of non-outliers**
- Replace below P1 with **min of non-outliers**

**Evaluation score:**  
`score = 100 × (1 − outliers_remaining / total_outliers)`  
→ Score = 100 when zero outliers remain ✅

In [ ]:
# ── Outlier replacement pipeline (competition solution) ─────────
import pandas as pd

# Simulate data.csv
rng = np.random.default_rng(42)
n   = 1000
data = pd.DataFrame({
    'order_id': range(1, n+1),
    'date':     pd.date_range('2023-01-01', periods=n, freq='h').strftime('%Y-%m-%d'),
    'revenue':  np.concatenate([
        rng.normal(500, 80, n-20),  # normal values
        rng.uniform(2000, 5000, 10),  # high outliers
        rng.uniform(-500, 0, 10),      # low outliers
    ])
})
# data = pd.read_csv('dataset/data.csv')  # ← use this in the actual competition

print(f'Before: shape={data.shape}  revenue range=[{data["revenue"].min():.0f}, {data["revenue"].max():.0f}]')

# ── Solution ───────────────────────────────────────────────────
# Step 1: percentile bounds
lower_bound = data['revenue'].quantile(0.01)
upper_bound = data['revenue'].quantile(0.99)

# Step 2: min/max of NON-outliers
mask_clean = (data['revenue'] >= lower_bound) & (data['revenue'] <= upper_bound)
rev_min = data.loc[mask_clean, 'revenue'].min()
rev_max = data.loc[mask_clean, 'revenue'].max()

outliers_before = (~mask_clean).sum()
print(f'P1={lower_bound:.1f}  P99={upper_bound:.1f}  '
      f'clean_min={rev_min:.1f}  clean_max={rev_max:.1f}')
print(f'Outliers found: {outliers_before}')

# Step 3: replace
submission = data.copy()
submission.loc[submission['revenue'] > upper_bound, 'revenue'] = rev_max
submission.loc[submission['revenue'] < lower_bound, 'revenue'] = rev_min

# submission.to_csv('submission.csv', index=False)  # ← save in competition

# Verify
still_outlier = ((submission['revenue'] < lower_bound) | (submission['revenue'] > upper_bound)).sum()
score = 100 * (1 - still_outlier / outliers_before) if outliers_before > 0 else 100
print(f'Outliers remaining: {still_outlier}  →  Score = {score:.0f}/100 ✅')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data['revenue'].hist(bins=50, ax=axes[0], color='crimson', alpha=0.7)
axes[0].set_title('Before: revenue with outliers')
submission['revenue'].hist(bins=50, ax=axes[1], color='steelblue', alpha=0.7)
axes[1].set_title('After: outliers replaced')
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

---
## 7 · Kafka Streaming ETL <a id='kafka'></a>

### Quiz question
> *"Correct sequence to build a Kafka stream-processing ETL pipeline?"*  
> ✅ **1 → 4 → 3 → 2**

| Step | Action | Role |
|---|---|---|
| **1** | Extract data into Kafka | Producer publishes to topic |
| **4** | Pull data from Kafka topics | Consumer subscribes and reads |
| **3** | Transform data in KStream objects | Real-time filtering/mapping/aggregation |
| **2** | Load data to other systems | Sink connector writes to DB/DWH |

```
[Source DB / App]
      ↓ (1) Extract
[Kafka Topic]  ←──── Kafka Broker (message queue)
      ↓ (4) Pull / Consume
[KStream Processor]
      ↓ (3) Transform (filter, map, aggregate)
[Sink / Target System]
      ↓ (2) Load
[Data Warehouse / DB / Dashboard]
```

### Key Kafka concepts

| Concept | Role |
|---|---|
| **Producer** | Publishes messages to a topic |
| **Consumer** | Subscribes to a topic and reads messages |
| **Topic** | Named channel — like a database table for streams |
| **Partition** | Parallelism unit within a topic |
| **KStream** | Continuous stream of records for real-time transformation |
| **KTable** | Stateful view — latest value per key |

---
## 8 · SQL in ETL — JOIN & Aggregation <a id='sql-etl'></a>

### Common ETL SQL patterns

```sql
-- Full name with uppercase last name + total spend, sorted
SELECT
    CONCAT(c.first_name, ' ', UPPER(c.last_name)) AS full_name,
    SUM(o.price)                                   AS amount_spent
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.id, c.first_name, c.last_name
ORDER BY amount_spent DESC,
         full_name    ASC;
```

In [ ]:
# ── SQL ETL pattern: customers + orders ────────────────────────
conn2 = sqlite3.connect(':memory:')

customers = pd.DataFrame({
    'id':         [1, 2, 3, 4],
    'first_name': ['Alice', 'Bob', 'Alice', 'Carol'],
    'last_name':  ['Smith', 'Jones', 'Adams', 'Brown'],
})
orders = pd.DataFrame({
    'order_id':    range(1, 9),
    'customer_id': [1, 1, 2, 2, 3, 4, 4, 4],
    'price':       [120, 80, 200, 150, 320, 50, 90, 180],
})

customers.to_sql('customers', conn2, index=False, if_exists='replace')
orders.to_sql('orders',    conn2, index=False, if_exists='replace')

result = pd.read_sql("""
    SELECT
        (c.first_name || ' ' || UPPER(c.last_name)) AS full_name,
        SUM(o.price)                                AS amount_spent
    FROM customers c
    JOIN orders o ON c.id = o.customer_id
    GROUP BY c.id, c.first_name, c.last_name
    ORDER BY amount_spent DESC,
             full_name    ASC
""", conn2)

display(result)

---
## 9 · Full Python ETL Pipeline <a id='full-pipeline'></a>

A complete, reusable ETL class pattern:

```
extract() → validate() → transform() → load() → test()
```

In [ ]:
# ── Full ETL Pipeline class ─────────────────────────────────────
import io, time

class ETLPipeline:
    """Reusable ETL pipeline with Extract, Transform, Load, and Test phases."""

    def __init__(self, name: str):
        self.name   = name
        self.source = None
        self.data   = None
        self.target = None

    # ── EXTRACT ──────────────────────────────────────────────────
    def extract(self, source_df: pd.DataFrame) -> 'ETLPipeline':
        self.source = source_df.copy()
        self.data   = source_df.copy()
        print(f'[EXTRACT] {len(self.data)} rows loaded')
        return self

    # ── TRANSFORM ────────────────────────────────────────────────
    def drop_nulls(self, cols) -> 'ETLPipeline':
        before = len(self.data)
        self.data.dropna(subset=cols, inplace=True)
        print(f'[TRANSFORM] drop_nulls: {before} → {len(self.data)} rows')
        return self

    def remove_duplicates(self, subset=None) -> 'ETLPipeline':
        before = len(self.data)
        self.data.drop_duplicates(subset=subset, inplace=True)
        print(f'[TRANSFORM] remove_duplicates: {before} → {len(self.data)} rows')
        return self

    def clip_outliers(self, col: str, p_low=0.01, p_high=0.99) -> 'ETLPipeline':
        lo = self.data[col].quantile(p_low)
        hi = self.data[col].quantile(p_high)
        clean_vals = self.data.loc[(self.data[col] >= lo) & (self.data[col] <= hi), col]
        self.data.loc[self.data[col] > hi, col] = clean_vals.max()
        self.data.loc[self.data[col] < lo, col] = clean_vals.min()
        print(f'[TRANSFORM] clip_outliers on {col}: P{p_low*100:.0f}={lo:.1f} P{p_high*100:.0f}={hi:.1f}')
        return self

    def add_column(self, name: str, func) -> 'ETLPipeline':
        self.data[name] = func(self.data)
        print(f'[TRANSFORM] added column: {name}')
        return self

    # ── INCREMENTAL FILTER ───────────────────────────────────────
    def incremental(self, date_col: str, last_run) -> 'ETLPipeline':
        before = len(self.data)
        self.data = self.data[pd.to_datetime(self.data[date_col]) > pd.Timestamp(last_run)]
        print(f'[EXTRACT] incremental: {before} → {len(self.data)} rows (after {last_run})')
        return self

    # ── LOAD ─────────────────────────────────────────────────────
    def load(self, conn, table_name: str) -> 'ETLPipeline':
        self.data.to_sql(table_name, conn, index=False, if_exists='replace')
        self.target = pd.read_sql(f'SELECT * FROM {table_name}', conn)
        print(f'[LOAD] {len(self.target)} rows written to {table_name}')
        return self

    # ── TEST ─────────────────────────────────────────────────────
    def test(self, id_col='order_id', value_col='revenue') -> bool:
        print('[TEST] Running data quality checks...')
        results = {
            'row_count_match':   len(self.source) >= len(self.target),
            'no_duplicates':     self.target[id_col].is_unique if id_col in self.target else True,
            'no_null_keys':      self.target[id_col].notna().all() if id_col in self.target else True,
            'no_null_values':    self.target[value_col].notna().all() if value_col in self.target else True,
        }
        for test, passed in results.items():
            print(f'  {test:30s}: {"✅" if passed else "❌"}')
        return all(results.values())


# ── Run the pipeline ───────────────────────────────────────────
rng  = np.random.default_rng(0)
n    = 200
raw  = pd.DataFrame({
    'order_id': range(1, n+1),
    'date':     pd.date_range('2024-01-01', periods=n, freq='h').strftime('%Y-%m-%d %H:%M'),
    'revenue':  np.concatenate([rng.normal(300, 50, n-10), rng.uniform(2000,5000,5), rng.uniform(-500,0,5)]),
    'customer': rng.choice(['Alice','Bob','Carol',None], n),  # some nulls
})
# Add a duplicate row
raw = pd.concat([raw, raw.iloc[[0]]], ignore_index=True)

target_conn = sqlite3.connect(':memory:')

pipeline = ETLPipeline('OrdersPipeline')
all_tests_pass = (
    pipeline
    .extract(raw)
    .drop_nulls(['customer'])
    .remove_duplicates(subset=['order_id'])
    .clip_outliers('revenue')
    .add_column('revenue_eur', lambda df: (df['revenue'] * 0.92).round(2))
    .load(target_conn, 'orders_clean')
    .test()
)

print(f'\nPipeline result: {"✅ SUCCESS" if all_tests_pass else "❌ FAILED"}')
display(pipeline.target.head(5))